# STEP 1: Load & Process DDL District Court Dataset

Source: Development Data Lab judicial data (`.dta` files).

This notebook is adapted from `01_load_ddl.py` and is designed to run on a Google Colab Python kernel.

## Colab usage notes
1. Upload the `dta/` folder (or mount Drive and point to it).
2. Install dependencies in the next cell.
3. Run cells in order.
4. Output parquet files are written to `compiled_dataset/`.

In [1]:
# Colab setup (run once per runtime)
# If already installed, this will be quick.
%pip install -q pyreadstat pyarrow pandas

import pandas as pd
import pyreadstat
import gc
from pathlib import Path

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 14.1 MB/s eta 0:00:0000:0100:01


In [11]:
# Connect Google Drive (Colab)
from google.colab import drive

drive.mount('/content/drive')
print('Connected to the drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Connected to the drive


In [3]:
# Configuration (optimized for ~12.67 GB RAM)
# Drive should be mounted in the previous cell.
DDL_ROOT = Path('/content/drive/MyDrive/MiniProject/dta')
OUTPUT_DIR = Path('compiled_dataset')
OUTPUT_DIR.mkdir(exist_ok=True)

# Process one year at a time.
YEARS = [2015]

# Chunk settings: lower this (e.g., 100_000) if runtime still struggles.
CHUNK_SIZE = 200_000

# Safer on low RAM: skip acts_sections merge by default because it is very large.
INCLUDE_ACTS_SECTIONS = False

In [4]:
def read_dta(path: Path, usecols=None) -> pd.DataFrame:
    """Read a Stata .dta file with optional column pruning."""
    if not path.exists():
        print(f"  [WARN] File not found: {path}")
        return pd.DataFrame()
    df, _ = pyreadstat.read_dta(str(path), usecols=usecols)
    return df


def load_keys() -> dict:
    """Load compact key tables with only the columns needed for merges."""
    root = Path(DDL_ROOT)
    keys = {}

    keys['act'] = read_dta(root / 'keys' / 'act_key.dta', usecols=['act', 'act_s'])
    keys['section'] = read_dta(root / 'keys' / 'section_key.dta', usecols=['section', 'section_s'])
    keys['disp_name'] = read_dta(root / 'keys' / 'disp_name_key.dta')
    keys['type_name'] = read_dta(root / 'keys' / 'type_name_key.dta')
    keys['state'] = read_dta(root / 'keys' / 'cases_state_key.dta')
    keys['district'] = read_dta(root / 'keys' / 'cases_district_key.dta')

    # Keep only useful columns where possible.
    if not keys['disp_name'].empty:
        keep = [c for c in ['disp_name', 'year', 'disp_name_s'] if c in keys['disp_name'].columns]
        keys['disp_name'] = keys['disp_name'][keep]
    if not keys['type_name'].empty:
        keep = [c for c in ['type_name', 'year', 'type_name_s'] if c in keys['type_name'].columns]
        keys['type_name'] = keys['type_name'][keep]

    if not keys['state'].empty:
        state_name_cols = [c for c in keys['state'].columns if 'state_name' in c or c.endswith('_s')]
        keep = [c for c in ['state_code', 'year'] if c in keys['state'].columns] + state_name_cols[:1]
        keys['state'] = keys['state'][keep]

    if not keys['district'].empty:
        dist_name_cols = [c for c in keys['district'].columns if 'dist_name' in c or c.endswith('_s')]
        keep = [c for c in ['state_code', 'dist_code', 'year'] if c in keys['district'].columns] + dist_name_cols[:1]
        keys['district'] = keys['district'][keep]

    return keys


def load_acts_sections() -> pd.DataFrame:
    """Optional large table load. Disabled by default for low-RAM runtimes."""
    if not INCLUDE_ACTS_SECTIONS:
        return pd.DataFrame()

    root = Path(DDL_ROOT)
    df = read_dta(
        root / 'acts_sections.dta',
        usecols=['ddl_case_id', 'act', 'section', 'bailable_ipc', 'criminal']
    )
    if df.empty:
        return df

    # Reduce one-to-many rows to one row per case id to control memory.
    agg_map = {}
    if 'act' in df.columns:
        agg_map['act'] = 'first'
    if 'section' in df.columns:
        agg_map['section'] = 'first'
    if 'bailable_ipc' in df.columns:
        agg_map['bailable_ipc'] = 'max'
    if 'criminal' in df.columns:
        agg_map['criminal'] = 'max'

    return df.groupby('ddl_case_id', as_index=False).agg(agg_map)


def iter_year_chunks(year: int):
    """Yield case data in chunks for a given year."""
    root = Path(DDL_ROOT)
    case_path = root / 'cases' / f'cases_{year}.dta'

    if not case_path.exists():
        print(f"  [WARN] Missing year file: {case_path}")
        return

    # Keep only columns needed downstream.
    wanted_cols = [
        'ddl_case_id', 'state_code', 'dist_code',
        'type_name', 'disp_name',
        'date_of_filing', 'date_of_decision',
        'pet_name', 'res_name'
    ]

    try:
        reader = pyreadstat.read_file_in_chunks(
            pyreadstat.read_dta,
            str(case_path),
            chunksize=CHUNK_SIZE,
            usecols=wanted_cols
        )
    except TypeError:
        # Fallback in case this pyreadstat build does not support usecols in chunk reader.
        reader = pyreadstat.read_file_in_chunks(
            pyreadstat.read_dta,
            str(case_path),
            chunksize=CHUNK_SIZE
        )

    for i, (chunk, _) in enumerate(reader, start=1):
        if chunk.empty:
            continue
        chunk['year'] = year
        yield i, chunk


def merge_chunk(chunk: pd.DataFrame, acts_df: pd.DataFrame, keys: dict) -> pd.DataFrame:
    """Apply lightweight merges to a single chunk."""
    df = chunk

    if not acts_df.empty and 'ddl_case_id' in df.columns:
        df = df.merge(acts_df, on='ddl_case_id', how='left')

    if not keys['act'].empty and 'act' in df.columns:
        df = df.merge(keys['act'], on='act', how='left', suffixes=('', '_key'))

    if not keys['section'].empty and 'section' in df.columns:
        df = df.merge(keys['section'], on='section', how='left', suffixes=('', '_key'))

    if not keys['disp_name'].empty and 'disp_name' in df.columns:
        merge_cols = [c for c in ['disp_name', 'year'] if c in keys['disp_name'].columns and c in df.columns]
        disp_cols = merge_cols + [c for c in ['disp_name_s'] if c in keys['disp_name'].columns]
        if merge_cols and len(disp_cols) > len(merge_cols):
            df = df.merge(keys['disp_name'][disp_cols], on=merge_cols, how='left')

    if not keys['type_name'].empty and 'type_name' in df.columns:
        merge_cols = [c for c in ['type_name', 'year'] if c in keys['type_name'].columns and c in df.columns]
        type_cols = merge_cols + [c for c in ['type_name_s'] if c in keys['type_name'].columns]
        if merge_cols and len(type_cols) > len(merge_cols):
            df = df.merge(keys['type_name'][type_cols], on=merge_cols, how='left')

    if not keys['state'].empty and 'state_code' in df.columns:
        state_cols = [c for c in ['state_code', 'year'] if c in keys['state'].columns and c in df.columns]
        state_name_cols = [c for c in keys['state'].columns if c not in ['state_code', 'year']]
        if state_cols and state_name_cols:
            df = df.merge(keys['state'][state_cols + state_name_cols[:1]], on=state_cols, how='left')

    if not keys['district'].empty and 'dist_code' in df.columns:
        dist_cols = [c for c in ['state_code', 'dist_code', 'year'] if c in keys['district'].columns and c in df.columns]
        dist_name_cols = [c for c in keys['district'].columns if c not in ['state_code', 'dist_code', 'year']]
        if dist_cols and dist_name_cols:
            df = df.merge(keys['district'][dist_cols + dist_name_cols[:1]], on=dist_cols, how='left')

    return df

In [5]:
FINAL_COLS = [
    'ddl_case_id', 'year', 'state_code', 'dist_code',
    'type_name_s', 'act_s', 'section_s', 'disp_name_s',
    'criminal', 'bailable_ipc',
    'state_name',
    'date_of_filing', 'date_of_decision',
    'pet_name', 'res_name',
]


def select_columns(df: pd.DataFrame) -> pd.DataFrame:
    available = [c for c in FINAL_COLS if c in df.columns]
    extra_string_cols = [c for c in df.columns if c.endswith('_s') and c not in available]
    all_keep = list(dict.fromkeys(available + extra_string_cols))
    return df[all_keep]


def optimize_chunk(df: pd.DataFrame) -> pd.DataFrame:
    # Trim whitespace in text columns.
    str_cols = df.select_dtypes('object').columns
    for col in str_cols:
        df[col] = df[col].str.strip()

    # Parse dates carefully.
    for datecol in ['date_of_filing', 'date_of_decision']:
        if datecol in df.columns:
            df[datecol] = pd.to_datetime(df[datecol], errors='coerce')

    # Downcast integer-like columns.
    for col in ['year', 'state_code', 'dist_code', 'criminal', 'bailable_ipc']:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce', downcast='integer')

    # Convert low-cardinality text to category for memory savings.
    for col in str_cols:
        non_null = df[col].notna().sum()
        if non_null == 0:
            continue
        uniq_ratio = df[col].nunique(dropna=True) / max(non_null, 1)
        if uniq_ratio < 0.10:
            df[col] = df[col].astype('category')

    return df


def run_pipeline(years=None):
    years = years or YEARS

    print('=' * 60)
    print('DDL District Court Data Loader (Low-RAM Mode)')
    print('=' * 60)
    print(f'CHUNK_SIZE={CHUNK_SIZE:,}, INCLUDE_ACTS_SECTIONS={INCLUDE_ACTS_SECTIONS}')

    print('\n[1] Loading key/lookup tables ...')
    keys = load_keys()
    print('    Keys loaded:', {k: len(v) for k, v in keys.items() if not v.empty})

    print('\n[2] Loading acts_sections table ...')
    acts_df = load_acts_sections()
    print(f"    Acts/sections rows kept: {len(acts_df):,}")

    print('\n[3] Processing each year in chunks ...')
    for year in years:
        print(f"\n  Processing {year} ...")
        total_rows = 0
        part_paths = []

        for chunk_idx, chunk in iter_year_chunks(year):
            df = merge_chunk(chunk, acts_df, keys)
            df = select_columns(df)
            df = optimize_chunk(df)

            out_path = OUTPUT_DIR / f'ddl_processed_{year}_part_{chunk_idx:04d}.parquet'
            df.to_parquet(out_path, index=False)
            part_paths.append(out_path)
            total_rows += len(df)

            mem_usage = df.memory_usage(deep=True).sum() / 1e9
            print(f"    chunk {chunk_idx:04d}: {len(df):,} rows, {mem_usage:.2f} GB -> {out_path.name}")

            del df
            gc.collect()

        if part_paths:
            print(f"  Year {year} complete: {total_rows:,} rows across {len(part_paths)} parquet part files.")
        else:
            print(f"  No chunks processed for {year}.")

    print('\n[4] Done.')
    print("Part files pattern: compiled_dataset/ddl_processed_<year>_part_*.parquet")
    print("To combine later (carefully):")
    print("pd.concat([pd.read_parquet(p) for p in sorted(OUTPUT_DIR.glob('ddl_processed_2015_part_*.parquet'))], ignore_index=True)")

In [6]:
# Run the pipeline
run_pipeline(YEARS)

DDL District Court Data Loader (Low-RAM Mode)
CHUNK_SIZE=200,000, INCLUDE_ACTS_SECTIONS=False

[1] Loading key/lookup tables ...
    Keys loaded: {'act': 29857, 'section': 2113919, 'disp_name': 462, 'type_name': 62714, 'state': 287, 'district': 632}

[2] Loading acts_sections table ...
    Acts/sections rows kept: 0

[3] Processing each year in chunks ...

  Processing 2015 ...
    chunk 0001: 200,000 rows, 0.02 GB -> ddl_processed_2015_part_0001.parquet
    chunk 0002: 200,000 rows, 0.02 GB -> ddl_processed_2015_part_0002.parquet
    chunk 0003: 200,000 rows, 0.02 GB -> ddl_processed_2015_part_0003.parquet
    chunk 0004: 200,000 rows, 0.02 GB -> ddl_processed_2015_part_0004.parquet
    chunk 0005: 200,000 rows, 0.02 GB -> ddl_processed_2015_part_0005.parquet
    chunk 0006: 200,000 rows, 0.02 GB -> ddl_processed_2015_part_0006.parquet
    chunk 0007: 200,000 rows, 0.02 GB -> ddl_processed_2015_part_0007.parquet
    chunk 0008: 200,000 rows, 0.02 GB -> ddl_processed_2015_part_0008.par

In [7]:
# Upload /content/compiled_dataset to Google Drive
from google.colab import drive
from pathlib import Path
import shutil

# 1) Mount Drive
drive.mount('/content/drive')

# 2) Define source (Colab kernel storage) and destination (Drive)
src = Path('/content/compiled_dataset')
dst = Path('/content/drive/MyDrive/MiniProject/compiled_dataset_180426')  # change if you want another folder

# 3) Validate source and copy
if not src.exists():
    raise FileNotFoundError(f"Source folder not found: {src}")

dst.parent.mkdir(parents=True, exist_ok=True)
shutil.copytree(src, dst, dirs_exist_ok=True)  # merge/overwrite existing files

print(f"Uploaded folder to: {dst}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Uploaded folder to: /content/drive/MyDrive/MiniProject/compiled_dataset_180426


In [17]:
# Combine all DDL part parquet files into one file and upload to Drive
import pyarrow.parquet as pq
from pathlib import Path
import shutil
DDL_ROOT = Path('/content/drive/MyDrive/MiniProject/compiled_dataset_180426')
OUTPUT_DIR = Path('/content/drive/MyDrive/MiniProject/compiled_dataset_180426')
OUTPUT_DIR.mkdir(exist_ok=True)
part_files = sorted(OUTPUT_DIR.glob('ddl_processed_2015_part_*.parquet'))
if not part_files:
    raise FileNotFoundError(f'No part files found in {OUTPUT_DIR}')

combined_path = OUTPUT_DIR / 'ddl_processed_2015.parquet'
writer = None
total_rows = 0

print(f'Found {len(part_files)} part files')
print('Combining in streaming mode...')

for i, part_path in enumerate(part_files, start=1):
    table = pq.read_table(part_path)
    if writer is None:
        writer = pq.ParquetWriter(combined_path, table.schema)
    writer.write_table(table)
    total_rows += table.num_rows

    if i % 10 == 0 or i == len(part_files):
        print(f'  merged {i}/{len(part_files)} parts')

if writer is not None:
    writer.close()

print(f'Combined file saved: {combined_path}')
print(f'Total rows written: {total_rows:,}')
print(f'File size: {combined_path.stat().st_size / 1e6:.1f} MB')

# Upload combined file to Drive folder
TARGET_DIR = Path('/content/drive/MyDrive/MiniProject/compiled_dataset_180426')
TARGET_DIR.mkdir(parents=True, exist_ok=True)
shutil.copy2(combined_path, TARGET_DIR / combined_path.name)
print(f'Uploaded: {combined_path.name} -> {TARGET_DIR}')

Found 53 part files
Combining in streaming mode...
  merged 10/53 parts
  merged 20/53 parts
  merged 30/53 parts
  merged 40/53 parts
  merged 50/53 parts
  merged 53/53 parts
Combined file saved: /content/drive/MyDrive/MiniProject/compiled_dataset_180426/ddl_processed_2015.parquet
Total rows written: 10,475,876
File size: 81.6 MB


SameFileError: PosixPath('/content/drive/MyDrive/MiniProject/compiled_dataset_180426/ddl_processed_2015.parquet') and PosixPath('/content/drive/MyDrive/MiniProject/compiled_dataset_180426/ddl_processed_2015.parquet') are the same file